In [0]:
incremental_path = '/Volumes/it_support_tickets/source/input_files/Incremental_file.csv'
df = spark.read.format('csv').option('header', 'true').option('inferSchema', 'true').load(incremental_path)
display(df)

In [0]:
from pyspark.sql.functions import current_timestamp, lit
incremental_df = df.withColumn('file_path', lit(incremental_path)) \
              .withColumn('ingestion_timestamp', current_timestamp()) \
              .withColumn('batch_id', lit(2))

In [0]:
incremental_df.count()

In [0]:
incremental_df.printSchema()

In [0]:
# combining all invalid conditions from cells 9 to 19
from pyspark.sql.functions import col

valid_engineers = [
    "Support Engineer 1",
    "Support Engineer 2",
    "Support Engineer 3",
    "Support Engineer 4",
    "Support Engineer 5",
    "Support Engineer 6"
]

valid_locations = [
    "Delhi",
    "Chennai",
    "Kochi",
    "Hyderabad",
    "Coimbatore",
    "Pune",
    "Mumbai",
    "Bengaluru"
]

invalid_condition = (
    col("created_date").isNull() |
    col("employee_id").isNull() |
    (~col("employee_id").rlike("^E[0-9]{3}$")) |
    col("support_category").isNull() |
    col("priority").isNull() |
    col("support_engineer").isNull() |
    (~col("support_engineer").isin(valid_engineers)) |
    col("location").isNull() |
    (~col("location").isin(valid_locations)) |
    col("resolution_hours").isNull() |
    (col("resolution_hours") <= 0) |
    col("hourly_cost").isNull() |
    (col("hourly_cost") <= 0) |
    col("ticket_status").isNull() |
    (col("ticket_status")== 'INVALID_STATUS')|
    (col("ticket_status") == "INVALID_VALUE")
)

display(invalid_condition)


In [0]:
incremental_bad = incremental_df.filter(invalid_condition)
display(incremental_bad)

In [0]:
incremental_good = incremental_df.filter(~invalid_condition)
display(incremental_good)


In [0]:
incremental_good.groupBy('ticket_id').count().filter('count > 1').orderBy('ticket_id').show()

In [0]:
# performing reconciliation
source_count = incremental_df.count()
good_df_count = incremental_good.count()
bad_df_count = incremental_bad.count()
print("Source Count: ", source_count)
print("Good Dataframe Count: ", good_df_count)
print("Bad Dataframe Count: ", bad_df_count)

if (source_count == (good_df_count + bad_df_count)):
    print("PASS")
else:
  print("FAIL")


In [0]:
incremental_good.write.mode('append').format('delta').option('mergeschema', 'true').saveAsTable('it_support_tickets.bronze.incremental_table')


In [0]:
# flagging each invalid incremental record with a rejection reason (same rules as batch 1)
from pyspark.sql.functions import when, col

incremental_bad_flagged = incremental_bad.withColumn(
    "rejection_reason",
    when(col("created_date").isNull(), "Invalid Date")
    .when(col("employee_id").isNull() | (~col("employee_id").rlike("^E[0-9]{3}$")), "Invalid Employee ID")
    .when(col("support_category").isNull(), "Invalid Support Category")
    .when(col("priority").isNull(), "Invalid Priority")
    .when(col("support_engineer").isNull() | (~col("support_engineer").isin(valid_engineers)), "Invalid Support Engineer")
    .when(col("location").isNull() | (~col("location").isin(valid_locations)), "Invalid Location")
    .when(col("resolution_hours").isNull() | (col("resolution_hours") <= 0), "Invalid Resolution Hours")
    .when(col("hourly_cost").isNull() | (col("hourly_cost") <= 0), "Invalid Hourly Cost")
    .when(col("ticket_status").isNull() | (col("ticket_status") == "INVALID_STATUS") | (col("ticket_status") == "INVALID_VALUE"), "Invalid Ticket Status")
    .otherwise(None)
)

display(incremental_bad_flagged)


In [0]:
# merging (upserting) the invalid incremental records into the existing quarantine table
from delta.tables import DeltaTable

quarantine_delta = DeltaTable.forName(spark, "it_support_tickets.bronze.quarantine_table")

(quarantine_delta.alias("target")
    .merge(
        incremental_bad_flagged.alias("source"),
        "target.ticket_id = source.ticket_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute())

In [0]:
# verifying the merge: quarantine_table should now hold bad records from both batches
quarantine_check = spark.read.table("it_support_tickets.bronze.quarantine_table")
display(quarantine_check.groupBy("batch_id", "rejection_reason").count().orderBy("batch_id", "rejection_reason"))
print("Total rows in quarantine_table:", quarantine_check.count())

In [0]:
# merging incremental batch (incremental_good, batch 2) into the initial bronze table
# so bronze_table holds records from both batches (56 from batch 1 + 37 from batch 2)

from delta.tables import DeltaTable

bronze_delta = DeltaTable.forName(spark, "it_support_tickets.bronze.bronze_table")

(bronze_delta.alias("target")
    .merge(
        incremental_good.alias("source"),
        "target.ticket_id = source.ticket_id AND target.batch_id = source.batch_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute())
# verifying the merge: bronze_table should now hold all records from both batches
# expected: 56 rows (batch 1) + 37 rows (batch 2) = 93 total
bronze_check = spark.read.table("it_support_tickets.bronze.bronze_table")
display(bronze_check.groupBy("batch_id").count().orderBy("batch_id"))
print("Total rows in bronze_table:", bronze_check.count())

batch1_count = bronze_check.filter("batch_id = 1").count()
batch2_count = bronze_check.filter("batch_id = 2").count()
print("Batch 1 rows in bronze_table:", batch1_count, "(expected 56)")
print("Batch 2 rows in bronze_table:", batch2_count, "(expected 37)")

# checking for duplicate merge keys (ticket_id, batch_id) - duplicates inflate the total
dup_keys = bronze_check.groupBy("ticket_id", "batch_id").count().filter("count > 1")
print("Duplicate (ticket_id, batch_id) keys in bronze_table:", dup_keys.count())
display(dup_keys.orderBy("batch_id", "ticket_id"))

# comparing actual source counts to find where any extra rows came from
print("Rows in incremental_good (batch 2 source):", incremental_good.count())
